In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [12]:
df_bronze = spark.read.json("s3a://raw/sptrans/posicoes/b720a7da-705f-4dae-9fad-fda8e78a3896")

TypeError: DataFrameReader.json() got an unexpected keyword argument 'HEADER'

In [11]:
df_bronze.show()

+-----+--------------------+
|   hr|                   l|
+-----+--------------------+
|19:46|[{373T-10, 1091, ...|
+-----+--------------------+



In [13]:
from pyspark.sql.functions import explode, col

# Explode a lista de linhas ("l") para que cada item seja uma linha do DataFrame
df_linhas = df_bronze.select(
    col("hr"),
    explode(col("l")).alias("linha")
)

In [14]:
df_linhas_selecionadas = df_linhas.select(
    "hr",
    col("linha.c").alias("codigo_linha"),
    col("linha.cl").alias("id_linha"),
    col("linha.lt0").alias("terminal_origem"),
    col("linha.lt1").alias("terminal_destino"),
    col("linha.sl").alias("sentido"),
    explode(col("linha.vs")).alias("veiculo")  # cada veículo vira uma linha
)

In [15]:
df_veiculos = df_linhas_selecionadas.select(
    "hr",
    "codigo_linha",
    "id_linha",
    "terminal_origem",
    "terminal_destino",
    "sentido",
    col("veiculo.p").alias("prefixo"),
    col("veiculo.a").alias("operando"),
    col("veiculo.ta").alias("atualizacao"),
    col("veiculo.px").alias("longitude"),
    col("veiculo.py").alias("latitude")
)

In [16]:
df_veiculos.show(20, truncate=False)

+-----+------------+--------+-----------------+-----------------+-------+-------+--------+--------------------+-------------------+-------------------+
|hr   |codigo_linha|id_linha|terminal_origem  |terminal_destino |sentido|prefixo|operando|atualizacao         |longitude          |latitude           |
+-----+------------+--------+-----------------+-----------------+-------+-------+--------+--------------------+-------------------+-------------------+
|19:46|373T-10     |1091    |METRÔ BRESSER    |JD. ITÁPOLIS     |1      |55331  |true    |2025-10-20T22:45:41Z|-46.532816         |-23.58748325       |
|19:46|373T-10     |1091    |METRÔ BRESSER    |JD. ITÁPOLIS     |1      |55312  |true    |2025-10-20T22:45:45Z|-46.498469         |-23.593679         |
|19:46|373T-10     |1091    |METRÔ BRESSER    |JD. ITÁPOLIS     |1      |55055  |true    |2025-10-20T22:46:04Z|-46.599900500000004|-23.554367624999998|
|19:46|373T-10     |1091    |METRÔ BRESSER    |JD. ITÁPOLIS     |1      |55008  |true   